In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
======================================================================
CONTROLLED ABLATION EXPERIMENT - CORRECTED VERSION
Hardware-Aligned Bit-Parallel Conflict Evaluation for SLS / 3-SAT
======================================================================

Purpose
-------
Controlled comparison of:

    1. Scalar conflict evaluation: O(k) per candidate
    2. Bit-parallel conflict evaluation: O(ceil(3k/64)) per candidate

The two implementations use EXACTLY the same:

    * Canonical state: selected[clause] = literal_index
    * Restart configurations (paired by instance)
    * Random search seed
    * Min-Conflict decision rule
    * Step limit

The ONLY algorithmic difference is candidate conflict evaluation:

    Scalar:   scan selected literals → O(k)
    Bit:      AND + POPCNT on masks → O(ceil(3k/64))

This is the exact operation from the paper:

    E(L) = sum_w popcnt(state_bits[w] & mask[L, w])

======================================================================
"""

import os
import sys
import json
import time
import random
from datetime import timedelta

import numpy as np
import matplotlib.pyplot as plt

from numba import njit, uint64, types
from scipy import stats


# =====================================================================
# USER CONFIGURATION
# =====================================================================

FOLDER_PATH = r"C:\Users\PC\Desktop\JSA\uf20-91"

MAX_FILES = 1000

MAX_STEPS = 5000
MAX_ATTEMPTS = 2000

BASE_SEED = 42

# Number of instances used for trajectory verification.
TRAJECTORY_TEST_FILES = 10

# Bootstrap repetitions.
N_BOOTSTRAP = 10000

# Output files.
RESULT_JSON = "ablation_results_final.json"
OUTPUT_DIR = "jsa_ablation_plots"

# Optional microbenchmark.
RUN_MICROBENCHMARK = True
MICROBENCH_EVALS = 50000


# =====================================================================
# PLOTTING CONFIGURATION
# =====================================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["pdf.fonttype"] = 42


# =====================================================================
# BASIC BIT OPERATIONS
# =====================================================================

@njit(inline="always")
def set_bit(arr, idx):
    word = idx >> 6
    bit = idx & 63
    arr[word] |= uint64(1) << uint64(bit)


@njit(inline="always")
def clear_bit(arr, idx):
    word = idx >> 6
    bit = idx & 63
    arr[word] &= ~(uint64(1) << uint64(bit))


@njit(inline="always")
def popcount_kernighan(value):
    """
    Portable 64-bit population count.

    NOTE: This is Kernighan's algorithm. The script checks whether
    Numba compiles this to hardware POPCNT. If the generated assembly
    does NOT contain POPCNT, do not claim hardware POPCNT in the paper.
    """
    count = uint64(0)
    v = value
    while v != 0:
        v &= v - uint64(1)
        count += uint64(1)
    return count


# =====================================================================
# CNF INSTANCE
# =====================================================================

class SATInstance:
    """
    3-SAT instance represented by a K x 3 literal grid.

    Canonical literal indexing:
        clause c: 3*c, 3*c+1, 3*c+2
    """

    def __init__(self, cnf_path):
        self.cnf_path = cnf_path
        self.clauses = []
        self.flat_lits = None
        self.num_clauses = 0
        self.num_lits = 0
        self.word_count = 0
        self.masks = None

        self.load_cnf()

        self.num_clauses = len(self.clauses)
        if self.num_clauses == 0:
            raise ValueError(f"No clauses found in {self.cnf_path}")

        # Verify 3-SAT
        for i, clause in enumerate(self.clauses):
            if len(clause) != 3:
                raise ValueError(
                    f"{self.cnf_path}: clause {i} has {len(clause)} literals; "
                    f"expected exactly 3."
                )

        self.num_lits = 3 * self.num_clauses
        self.word_count = (self.num_lits + 63) // 64

        self.flat_lits = np.asarray(
            [lit for clause in self.clauses for lit in clause],
            dtype=np.int32
        )

        self._precompute_masks()

    # -----------------------------------------------------------------
    # CNF parser
    # -----------------------------------------------------------------

    def load_cnf(self):
        """Robust DIMACS parser supporting clauses spanning multiple lines."""
        current_clause = []

        with open(self.cnf_path, "r") as f:
            for raw_line in f:
                line = raw_line.strip()
                if not line:
                    continue
                if line.startswith("c") or line.startswith("p") or line.startswith("%"):
                    continue

                tokens = line.split()
                for token in tokens:
                    try:
                        value = int(token)
                    except ValueError:
                        continue

                    if value == 0:
                        if current_clause:
                            self.clauses.append(current_clause)
                            current_clause = []
                    else:
                        current_clause.append(value)

        if current_clause:
            self.clauses.append(current_clause)

    # -----------------------------------------------------------------
    # Contradiction masks
    # -----------------------------------------------------------------

    def _precompute_masks(self):
        """
        mask[literal_index, word]

        A bit is set iff the corresponding cell contains a literal
        contradictory to the selected literal.

        Same-clause literals are excluded because a configuration
        selects exactly one literal per clause.
        """
        self.masks = np.zeros(
            (self.num_lits, self.word_count),
            dtype=np.uint64
        )

        for i in range(self.num_lits):
            clause_i = i // 3
            literal_i = int(self.flat_lits[i])

            for j in range(self.num_lits):
                clause_j = j // 3
                if clause_i == clause_j:
                    continue

                literal_j = int(self.flat_lits[j])
                if literal_i == -literal_j:
                    word = j >> 6
                    bit = j & 63
                    self.masks[i, word] |= (np.uint64(1) << np.uint64(bit))

    # -----------------------------------------------------------------
    # Initial configuration
    # -----------------------------------------------------------------

    def initialize_state(self, seed):
        """
        Generate one canonical configuration.
        Exactly one literal is selected from each clause.
        """
        rng = random.Random(seed)
        selected = np.empty(self.num_clauses, dtype=np.int32)
        for c in range(self.num_clauses):
            selected[c] = 3 * c + rng.randrange(3)
        return selected

    # -----------------------------------------------------------------
    # Packed representation
    # -----------------------------------------------------------------

    def selected_to_bits(self, selected):
        bits = np.zeros(self.word_count, dtype=np.uint64)
        for c in range(self.num_clauses):
            idx = int(selected[c])
            word = idx >> 6
            bit = idx & 63
            bits[word] |= np.uint64(1) << np.uint64(bit)
        return bits


# =====================================================================
# SCALAR EVALUATORS - O(k) per candidate
# =====================================================================

@njit
def scalar_total_energy(selected, flat_lits, num_clauses):
    """
    Total number of contradictory selected-literal pairs.
    Each pair is counted exactly once.
    """
    conflicts = 0
    for c1 in range(num_clauses):
        lit1 = flat_lits[selected[c1]]
        for c2 in range(c1 + 1, num_clauses):
            lit2 = flat_lits[selected[c2]]
            if lit1 == -lit2:
                conflicts += 1
    return conflicts


@njit
def scalar_clause_conflicts(selected, flat_lits, num_clauses, c_idx):
    """
    Number of selected literals contradicting the literal currently
    selected in clause c_idx.

    Complexity: O(k)
    """
    selected_literal = flat_lits[selected[c_idx]]
    conflicts = 0

    for d in range(num_clauses):
        if d == c_idx:
            continue
        other_literal = flat_lits[selected[d]]
        if selected_literal == -other_literal:
            conflicts += 1

    return conflicts


@njit
def scalar_candidate_conflicts(selected, flat_lits, num_clauses, c_idx, cand_idx):
    """
    Number of currently selected literals contradicting candidate L.

    Complexity: O(k)

    This is the scalar counterpart of the bit-parallel mask kernel.
    """
    candidate_literal = flat_lits[cand_idx]
    conflicts = 0

    for d in range(num_clauses):
        if d == c_idx:
            continue
        selected_literal = flat_lits[selected[d]]
        if candidate_literal == -selected_literal:
            conflicts += 1

    return conflicts


# =====================================================================
# BIT-PARALLEL EVALUATORS - O(ceil(3k/64)) per candidate
# =====================================================================

@njit
def bit_clause_conflicts(selected, state_bits, masks, word_count, c_idx):
    """
    Number of selected literals contradicting the currently selected
    literal in clause c_idx.

    Complexity: O(ceil(3k/64))

    Uses the maintained state_bits, does NOT rebuild.
    """
    literal_index = selected[c_idx]
    conflicts = uint64(0)

    for w in range(word_count):
        intersection = state_bits[w] & masks[literal_index, w]
        conflicts += popcount_kernighan(intersection)

    return conflicts


@njit
def bit_candidate_conflicts(state_bits, masks, word_count, cand_idx):
    """
    Bit-parallel candidate conflict evaluation.

    Complexity: O(ceil(3k/64))

    This is the core hardware-aligned kernel from the paper:
        E(L) = sum_w popcnt(state_bits[w] & mask[L, w])
    """
    conflicts = uint64(0)

    for w in range(word_count):
        intersection = state_bits[w] & masks[cand_idx, w]
        conflicts += popcount_kernighan(intersection)

    return conflicts


@njit
def bit_total_energy(selected, state_bits, masks, word_count, num_clauses):
    """
    Total contradictory pair count.

    Each contradictory pair is observed twice:
        L_i sees ¬L_j
        L_j sees ¬L_i

    Therefore divide by two.
    """
    total = uint64(0)

    for c in range(num_clauses):
        literal_index = selected[c]
        for w in range(word_count):
            intersection = state_bits[w] & masks[literal_index, w]
            total += popcount_kernighan(intersection)

    return total // uint64(2)


# =====================================================================
# VALIDATION: EQUIVALENCE OF SCALAR AND BIT OBJECTIVES
# =====================================================================

@njit
def validate_state_energy(selected, flat_lits, masks, state_bits,
                          num_clauses, word_count):
    """Return True if scalar and bit total energies are identical."""
    scalar_e = scalar_total_energy(selected, flat_lits, num_clauses)
    bit_e = bit_total_energy(selected, state_bits, masks, word_count, num_clauses)
    return scalar_e == bit_e


@njit
def validate_candidate_evaluations(selected, flat_lits, masks, state_bits,
                                   num_clauses, word_count):
    """Verify every literal candidate in every clause."""
    for c in range(num_clauses):
        start = 3 * c
        for candidate in range(start, start + 3):
            scalar_val = scalar_candidate_conflicts(
                selected, flat_lits, num_clauses, c, candidate
            )
            bit_val = bit_candidate_conflicts(
                state_bits, masks, word_count, candidate
            )
            if scalar_val != bit_val:
                return False
    return True


# =====================================================================
# SEARCH ENGINE: SCALAR
# =====================================================================

@njit
def solve_scalar(initial_states, flat_lits, num_clauses,
                 max_steps, max_attempts, search_seed,
                 record_trajectory):
    """
    Scalar Min-Conflict solver.

    initial_states: matrix [max_attempts, num_clauses]
        Each restart receives a predetermined initial configuration.

    This guarantees that scalar and bit solvers receive identical
    restart states.
    """
    np.random.seed(search_seed)

    selected = np.empty(num_clauses, dtype=np.int32)

    total_flips = 0
    attempts_used = 0
    final_steps = 0

    max_record = max_attempts * max_steps
    trajectory = np.empty((max_record, num_clauses), dtype=np.int32)
    trajectory_length = 0

    for attempt in range(max_attempts):
        attempts_used += 1

        # Identical predetermined restart state
        for c in range(num_clauses):
            selected[c] = initial_states[attempt, c]

        for step in range(max_steps):
            energy = scalar_total_energy(selected, flat_lits, num_clauses)

            # Success BEFORE a flip
            if energy == 0:
                final_steps = step
                return (True, total_flips, attempts_used, final_steps,
                        trajectory, trajectory_length)

            # Find conflicted clauses
            conflicted_count = 0
            conflicted = np.empty(num_clauses, dtype=np.int32)

            for c in range(num_clauses):
                if scalar_clause_conflicts(selected, flat_lits, num_clauses, c) > 0:
                    conflicted[conflicted_count] = c
                    conflicted_count += 1

            if conflicted_count == 0:
                break

            # Random conflicted clause
            c_idx = conflicted[np.random.randint(0, conflicted_count)]

            # Min-Conflict candidate selection
            current_literal = selected[c_idx]
            best_literal = current_literal
            best_conflicts = scalar_candidate_conflicts(
                selected, flat_lits, num_clauses, c_idx, current_literal
            )

            row_start = 3 * c_idx
            for candidate in range(row_start, row_start + 3):
                if candidate == current_literal:
                    continue

                cand_conflicts = scalar_candidate_conflicts(
                    selected, flat_lits, num_clauses, c_idx, candidate
                )

                if cand_conflicts < best_conflicts:
                    best_conflicts = cand_conflicts
                    best_literal = candidate

                if best_conflicts == 0:
                    break

            # Apply move
            if best_literal != current_literal:
                selected[c_idx] = best_literal
                total_flips += 1

            # Record trajectory AFTER move
            if record_trajectory and trajectory_length < max_record:
                for c in range(num_clauses):
                    trajectory[trajectory_length, c] = selected[c]
                trajectory_length += 1

    return (False, total_flips, attempts_used, 0,
            trajectory, trajectory_length)


# =====================================================================
# SEARCH ENGINE: BIT-PARALLEL
# =====================================================================

@njit
def solve_bit(initial_states, masks, num_clauses, word_count,
              max_steps, max_attempts, search_seed,
              record_trajectory):
    """
    Bit-parallel Min-Conflict solver.

    The search logic is intentionally identical to solve_scalar().
    Only conflict evaluation changes.

    Maintains state_bits incrementally for O(k/64) candidate evaluation.
    """
    np.random.seed(search_seed)

    selected = np.empty(num_clauses, dtype=np.int32)
    state_bits = np.zeros(word_count, dtype=np.uint64)

    total_flips = 0
    attempts_used = 0
    final_steps = 0

    max_record = max_attempts * max_steps
    trajectory = np.empty((max_record, num_clauses), dtype=np.int32)
    trajectory_length = 0

    for attempt in range(max_attempts):
        attempts_used += 1

        # Identical predetermined restart state
        for c in range(num_clauses):
            selected[c] = initial_states[attempt, c]

        # Rebuild state_bits ONCE per restart.
        for w in range(word_count):
            state_bits[w] = uint64(0)

        for c in range(num_clauses):
            literal_index = selected[c]
            word = literal_index >> 6
            bit = literal_index & 63
            state_bits[word] |= uint64(1) << uint64(bit)

        for step in range(max_steps):
            energy = bit_total_energy(selected, state_bits, masks,
                                      word_count, num_clauses)

            # Success BEFORE a flip
            if energy == 0:
                final_steps = step
                return (True, total_flips, attempts_used, final_steps,
                        trajectory, trajectory_length)

            # Find conflicted clauses
            conflicted_count = 0
            conflicted = np.empty(num_clauses, dtype=np.int32)

            for c in range(num_clauses):
                if bit_clause_conflicts(selected, state_bits, masks, word_count, c) > 0:
                    conflicted[conflicted_count] = c
                    conflicted_count += 1

            if conflicted_count == 0:
                break

            # Random conflicted clause (same as scalar)
            c_idx = conflicted[np.random.randint(0, conflicted_count)]

            # Min-Conflict candidate selection
            current_literal = selected[c_idx]
            best_literal = current_literal
            best_conflicts = bit_candidate_conflicts(
                state_bits, masks, word_count, current_literal
            )

            row_start = 3 * c_idx
            for candidate in range(row_start, row_start + 3):
                if candidate == current_literal:
                    continue

                cand_conflicts = bit_candidate_conflicts(
                    state_bits, masks, word_count, candidate
                )

                if cand_conflicts < best_conflicts:
                    best_conflicts = cand_conflicts
                    best_literal = candidate

                if best_conflicts == 0:
                    break

            # Apply move with incremental state_bits update
            if best_literal != current_literal:
                # Remove old selected cell
                old_word = current_literal >> 6
                old_bit = current_literal & 63
                state_bits[old_word] &= ~(uint64(1) << uint64(old_bit))

                # Insert new selected cell
                new_word = best_literal >> 6
                new_bit = best_literal & 63
                state_bits[new_word] |= uint64(1) << uint64(new_bit)

                selected[c_idx] = best_literal
                total_flips += 1

            # Record trajectory AFTER move
            if record_trajectory and trajectory_length < max_record:
                for c in range(num_clauses):
                    trajectory[trajectory_length, c] = selected[c]
                trajectory_length += 1

    return (False, total_flips, attempts_used, 0,
            trajectory, trajectory_length)


# =====================================================================
# RESTART CONFIGURATION GENERATOR
# =====================================================================

def generate_restart_states(instance, base_seed, max_attempts):
    """
    Generate ALL restart configurations once.

    Both scalar and bit solvers receive exactly the same matrix.
    Each restart uses a DIFFERENT initial configuration.
    """
    states = np.empty((max_attempts, instance.num_clauses), dtype=np.int32)

    for attempt in range(max_attempts):
        restart_seed = base_seed + 1000003 * (attempt + 1)
        states[attempt] = instance.initialize_state(restart_seed)

    return states


# =====================================================================
# WARM-UP / JIT COMPILATION
# =====================================================================

def warmup_instance(instance):
    """Compile all Numba kernels before measurements."""
    k = instance.num_clauses
    m = instance.word_count

    selected = instance.initialize_state(12345)
    state_bits = instance.selected_to_bits(selected)

    selected_matrix = np.empty((1, k), dtype=np.int32)
    selected_matrix[0] = selected

    # Scalar kernels
    scalar_total_energy(selected, instance.flat_lits, k)
    scalar_clause_conflicts(selected, instance.flat_lits, k, 0)
    scalar_candidate_conflicts(selected, instance.flat_lits, k, 0, selected[0])

    # Bit kernels
    bit_total_energy(selected, state_bits, instance.masks, m, k)
    bit_clause_conflicts(selected, state_bits, instance.masks, m, 0)
    bit_candidate_conflicts(state_bits, instance.masks, m, selected[0])

    # Complete solver compilation
    solve_scalar(selected_matrix, instance.flat_lits, k, 1, 1, 42, False)
    solve_bit(selected_matrix, instance.masks, k, m, 1, 1, 42, False)


# =====================================================================
# POPCNT ASSEMBLY VERIFICATION
# =====================================================================

def verify_popcnt_assembly():
    """
    Inspect Numba-generated assembly for POPCNT.

    Returns:
        dictionary with verification information.

    This is deliberately conservative: if the assembly cannot be
    inspected or does not contain POPCNT, we do NOT claim hardware POPCNT.
    """
    result = {
        "assembly_available": False,
        "popcnt_detected": False,
        "message": ""
    }

    try:
        # Force compilation for uint64
        popcount_kernighan(np.uint64(0x123456789ABCDEF0))

        asm = popcount_kernighan.inspect_asm((types.uint64,))
        result["assembly_available"] = True

        asm_lower = asm.lower()
        detected = "popcnt" in asm_lower

        result["popcnt_detected"] = detected

        if detected:
            result["message"] = (
                "The generated Numba assembly contains a POPCNT instruction."
            )
        else:
            result["message"] = (
                "No POPCNT mnemonic was detected in the generated assembly. "
                "Do not describe this run as hardware-POPCNT verified."
            )

    except Exception as exc:
        result["message"] = f"Assembly inspection failed: {repr(exc)}"

    return result


# =====================================================================
# TRAJECTORY VERIFICATION
# =====================================================================

def verify_trajectory(instance, initial_states, search_seed,
                      max_steps=100, max_attempts=20):
    """
    Verify that scalar and bit implementations produce identical
    trajectories.

    Intended for small validation experiments, not the large benchmark.
    """
    limited_states = initial_states[:max_attempts].copy()

    scalar_result = solve_scalar(
        limited_states, instance.flat_lits, instance.num_clauses,
        max_steps, max_attempts, search_seed, True
    )

    bit_result = solve_bit(
        limited_states, instance.masks, instance.num_clauses,
        instance.word_count, max_steps, max_attempts, search_seed, True
    )

    scalar_success = scalar_result[0]
    bit_success = bit_result[0]
    scalar_length = scalar_result[5]
    bit_length = bit_result[5]

    if scalar_length != bit_length:
        return {
            "identical": False,
            "reason": f"trajectory lengths differ: {scalar_length} vs {bit_length}"
        }

    if scalar_success != bit_success:
        return {
            "identical": False,
            "reason": f"success status differs: {scalar_success} vs {bit_success}"
        }

    scalar_trajectory = scalar_result[4][:scalar_length]
    bit_trajectory = bit_result[4][:bit_length]

    identical = np.array_equal(scalar_trajectory, bit_trajectory)

    if identical:
        return {"identical": True, "reason": "Trajectories are identical.", "length": scalar_length}

    # Locate first mismatch
    mismatch = np.argwhere(scalar_trajectory != bit_trajectory)
    if mismatch.size > 0:
        first_row = int(mismatch[0][0])
        first_col = int(mismatch[0][1])
        reason = (
            f"first mismatch at trajectory step {first_row}, clause {first_col}: "
            f"scalar={scalar_trajectory[first_row, first_col]}, "
            f"bit={bit_trajectory[first_row, first_col]}"
        )
    else:
        reason = "Unknown trajectory mismatch."

    return {"identical": False, "reason": reason, "length": scalar_length}


# =====================================================================
# MICROBENCHMARK
# =====================================================================

@njit
def scalar_microbenchmark(selected, flat_lits, num_clauses, repetitions):
    """Repeated scalar candidate evaluation."""
    accumulator = 0
    candidate = selected[0]

    for _ in range(repetitions):
        accumulator += scalar_candidate_conflicts(
            selected, flat_lits, num_clauses, 0, candidate
        )

    return accumulator


@njit
def bit_microbenchmark(state_bits, masks, word_count, candidate, repetitions):
    """Repeated bit-parallel candidate evaluation."""
    accumulator = uint64(0)

    for _ in range(repetitions):
        accumulator += bit_candidate_conflicts(
            state_bits, masks, word_count, candidate
        )

    return accumulator


def run_microbenchmark(instance, repetitions=50000):
    print("\n" + "=" * 72)
    print("PURE CANDIDATE-EVALUATION MICROBENCHMARK")
    print("=" * 72)

    selected = instance.initialize_state(987654)
    state_bits = instance.selected_to_bits(selected)
    candidate = int(selected[0])

    # Warm-up
    scalar_microbenchmark(selected, instance.flat_lits, instance.num_clauses, 10)
    bit_microbenchmark(state_bits, instance.masks, instance.word_count, candidate, 10)

    # Scalar
    t0 = time.perf_counter()
    scalar_microbenchmark(selected, instance.flat_lits, instance.num_clauses, repetitions)
    scalar_elapsed = time.perf_counter() - t0

    # Bit
    t0 = time.perf_counter()
    bit_microbenchmark(state_bits, instance.masks, instance.word_count, candidate, repetitions)
    bit_elapsed = time.perf_counter() - t0

    scalar_ns = scalar_elapsed / repetitions * 1e9
    bit_ns = bit_elapsed / repetitions * 1e9
    speedup = scalar_ns / bit_ns if bit_ns > 0 else float("inf")

    result = {
        "repetitions": repetitions,
        "scalar_ns_per_eval": scalar_ns,
        "bit_ns_per_eval": bit_ns,
        "speedup": speedup,
        "clauses": instance.num_clauses,
        "words": instance.word_count
    }

    print(f"Clauses:              {instance.num_clauses}")
    print(f"64-bit words:         {instance.word_count}")
    print(f"Scalar:               {scalar_ns:.2f} ns/eval")
    print(f"Bit-parallel:         {bit_ns:.2f} ns/eval")
    print(f"Speedup:              {speedup:.2f}x")

    return result


# =====================================================================
# MEMORY SCALING
# =====================================================================

def compute_memory_table():
    print("\n" + "=" * 72)
    print("MEMORY SCALING ANALYSIS (Capacity-Based Regimes)")
    print("=" * 72)
    print("NOTE: These are capacity-based estimates, not measured cache misses.")
    print("-" * 72)

    print(f"{'n':>8} {'k':>8} {'m':>8} {'Memory KB':>14} {'Nominal regime':>28}")
    print("-" * 72)

    test_cases = [
        (20, 91), (20, 200), (20, 400), (20, 1000), (20, 2000), (20, 5000), (20, 10000),
        (100, 425), (100, 850), (100, 2125),
        (200, 850), (200, 2125), (200, 4250),
        (500, 4250), (1000, 4250)
    ]

    rows = []

    for n, k in test_cases:
        m = (3 * k + 63) // 64
        memory_bytes = 2 * n * m * 8
        memory_kb = memory_bytes / 1024.0

        if memory_bytes <= 32 * 1024:
            regime = "L1 capacity"
        elif memory_bytes <= 256 * 1024:
            regime = "L2 capacity"
        elif memory_bytes <= 3 * 1024 * 1024:
            regime = "L3 capacity"
        else:
            regime = "Beyond L3"

        print(f"{n:>8} {k:>8} {m:>8} {memory_kb:>14.2f} {regime:>28}")

        rows.append({
            "variables": n,
            "clauses": k,
            "words": m,
            "memory_bytes": memory_bytes,
            "memory_kb": memory_kb,
            "capacity_regime": regime
        })

    print("\nFormula: M(n,k) = 2 n ceil(3k/64) × 8 bytes")
    print("\nNOTE: These are capacity-based regimes, not measured cache misses.")
    print("      Actual cache behavior requires hardware performance counters.")

    return rows


# =====================================================================
# RUN ONE INSTANCE
# =====================================================================

def run_one_instance(instance, restart_states, search_seed,
                     max_steps, max_attempts):
    """
    Execute both solvers on the same instance.

    Timing excludes CNF parsing, mask construction, restart-state generation,
    and JIT compilation. Timing includes search execution.
    """
    # Scalar
    t0 = time.perf_counter()
    result_s = solve_scalar(
        restart_states, instance.flat_lits, instance.num_clauses,
        max_steps, max_attempts, search_seed, False
    )
    scalar_time = time.perf_counter() - t0

    # Bit-parallel
    t0 = time.perf_counter()
    result_b = solve_bit(
        restart_states, instance.masks, instance.num_clauses,
        instance.word_count, max_steps, max_attempts, search_seed, False
    )
    bit_time = time.perf_counter() - t0

    scalar_success = bool(result_s[0])
    bit_success = bool(result_b[0])

    record = {
        "scalar_success": scalar_success,
        "bit_success": bit_success,
        "scalar_time": float(scalar_time),
        "bit_time": float(bit_time),
        "scalar_flips_total": int(result_s[1]),
        "bit_flips_total": int(result_b[1]),
        "scalar_attempts": int(result_s[2]),
        "bit_attempts": int(result_b[2]),
        "scalar_final_steps": int(result_s[3]),
        "bit_final_steps": int(result_b[3]),
        "same_success": (scalar_success == bit_success)
    }

    return record


# =====================================================================
# ABLATION EXPERIMENT
# =====================================================================

def run_ablation_experiment(folder_path, max_files=1000, max_steps=5000,
                            max_attempts=2000, base_seed=42,
                            trajectory_test_files=10):
    print("\n" + "=" * 72)
    print("CONTROLLED ABLATION EXPERIMENT")
    print("=" * 72)

    print("Canonical state: selected[clause] = literal index")
    print("Scalar candidate evaluation: O(k)")
    print("Bit candidate evaluation: O(ceil(3k/64))")
    print("Same restart states + same random search seed")
    print("Only conflict evaluation implementation changes")
    print("-" * 72)

    files = sorted(f for f in os.listdir(folder_path) if f.lower().endswith(".cnf"))
    files = files[:max_files]
    total_files = len(files)

    if total_files == 0:
        raise FileNotFoundError(f"No .cnf files found in {folder_path}")

    print(f"Found {total_files} CNF files.")
    print(f"Maximum steps/restart: {max_steps}")
    print(f"Maximum restarts: {max_attempts}")
    print("-" * 72)

    paired_results = []
    trajectory_checks = []
    popcnt_info = None

    start_benchmark = time.perf_counter()

    for idx, filename in enumerate(files):
        path = os.path.join(folder_path, filename)
        instance = SATInstance(path)

        # Compile first instance before timing benchmark
        if idx == 0:
            print("\nCompiling Numba kernels...")
            warmup_instance(instance)
            print("Numba warm-up complete.")

            popcnt_info = verify_popcnt_assembly()
            print("\nPOPCNT verification:")
            print(f"  Assembly available: {popcnt_info['assembly_available']}")
            print(f"  POPCNT detected: {popcnt_info['popcnt_detected']}")
            print(f"  {popcnt_info['message']}")

        # Same restart configurations for both solvers
        instance_seed = base_seed + idx
        restart_states = generate_restart_states(instance, instance_seed, max_attempts)

        search_seed = base_seed + 500000 + idx

        # Verify scalar/bit mathematical equivalence on initial state
        test_selected = restart_states[0].copy()
        test_bits = instance.selected_to_bits(test_selected)

        energy_ok = validate_state_energy(
            test_selected, instance.flat_lits, instance.masks, test_bits,
            instance.num_clauses, instance.word_count
        )

        candidate_ok = validate_candidate_evaluations(
            test_selected, instance.flat_lits, instance.masks, test_bits,
            instance.num_clauses, instance.word_count
        )

        if not energy_ok:
            raise RuntimeError(f"Energy mismatch detected in {filename}")

        if not candidate_ok:
            raise RuntimeError(f"Candidate evaluator mismatch detected in {filename}")

        # Optional trajectory validation
        if idx < trajectory_test_files:
            trajectory_result = verify_trajectory(
                instance, restart_states, search_seed,
                max_steps=min(max_steps, 100),
                max_attempts=min(max_attempts, 20)
            )

            trajectory_checks.append({
                "instance": filename,
                **trajectory_result
            })

            if not trajectory_result["identical"]:
                raise RuntimeError(
                    f"\nTrajectory mismatch detected.\n"
                    f"Instance: {filename}\n"
                    f"Reason: {trajectory_result['reason']}"
                )

        # Progress display
        progress = (idx + 1) / total_files
        elapsed = time.perf_counter() - start_benchmark

        if idx > 0:
            average = elapsed / (idx + 1)
            remaining = total_files - idx - 1
            eta_seconds = average * remaining
            eta = str(timedelta(seconds=int(eta_seconds)))
        else:
            eta = "calculating..."

        bar_length = 40
        filled = int(bar_length * progress)
        bar = "█" * filled + "░" * (bar_length - filled)

        sys.stdout.write(
            f"\rProgress |{bar}| {progress*100:6.2f}% "
            f"| {idx+1}/{total_files} "
            f"| {filename[:28]:<28} "
            f"| ETA {eta:<12}"
        )
        sys.stdout.flush()

        # Actual paired benchmark
        result = run_one_instance(instance, restart_states, search_seed,
                                  max_steps, max_attempts)

        record = {
            "instance": filename,
            "seed": instance_seed,
            "search_seed": search_seed,
            "num_clauses": instance.num_clauses,
            "word_count": instance.word_count,
            **result
        }

        paired_results.append(record)

    print("\n")

    return paired_results, trajectory_checks, popcnt_info


# =====================================================================
# STATISTICS
# =====================================================================

def bootstrap_mean_difference(scalar_times, bit_times, n_bootstrap=10000, seed=123456):
    """Paired bootstrap for scalar_time - bit_time."""
    scalar_times = np.asarray(scalar_times, dtype=np.float64)
    bit_times = np.asarray(bit_times, dtype=np.float64)
    differences = scalar_times - bit_times

    rng = np.random.default_rng(seed)
    n = len(differences)
    means = np.empty(n_bootstrap, dtype=np.float64)

    for i in range(n_bootstrap):
        sample = rng.choice(differences, size=n, replace=True)
        means[i] = np.mean(sample)

    return {
        "lower": float(np.percentile(means, 2.5)),
        "upper": float(np.percentile(means, 97.5)),
        "mean": float(np.mean(differences))
    }


def bootstrap_median_difference(scalar_times, bit_times, n_bootstrap=10000, seed=123457):
    """Paired bootstrap for median runtime difference."""
    scalar_times = np.asarray(scalar_times, dtype=np.float64)
    bit_times = np.asarray(bit_times, dtype=np.float64)
    differences = scalar_times - bit_times

    rng = np.random.default_rng(seed)
    n = len(differences)
    medians = np.empty(n_bootstrap, dtype=np.float64)

    for i in range(n_bootstrap):
        sample = rng.choice(differences, size=n, replace=True)
        medians[i] = np.median(sample)

    return {
        "lower": float(np.percentile(medians, 2.5)),
        "upper": float(np.percentile(medians, 97.5)),
        "median": float(np.median(differences))
    }


def compute_ablation_stats(paired_results, n_bootstrap=10000):
    total = len(paired_results)

    if total == 0:
        return {"error": "No benchmark results."}

    scalar_success_count = sum(1 for r in paired_results if r["scalar_success"])
    bit_success_count = sum(1 for r in paired_results if r["bit_success"])

    # Paired successful solves
    successful_pairs = [
        r for r in paired_results
        if r["scalar_success"] and r["bit_success"]
    ]

    if not successful_pairs:
        return {
            "error": "No successful paired runs.",
            "success_rates": {
                "scalar_count": scalar_success_count,
                "bit_count": bit_success_count,
                "total": total
            }
        }

    n_pairs = len(successful_pairs)

    scalar_times = np.asarray([r["scalar_time"] for r in successful_pairs], dtype=np.float64)
    bit_times = np.asarray([r["bit_time"] for r in successful_pairs], dtype=np.float64)
    scalar_flips = np.asarray([r["scalar_flips_total"] for r in successful_pairs], dtype=np.float64)
    bit_flips = np.asarray([r["bit_flips_total"] for r in successful_pairs], dtype=np.float64)
    scalar_attempts = np.asarray([r["scalar_attempts"] for r in successful_pairs], dtype=np.float64)
    bit_attempts = np.asarray([r["bit_attempts"] for r in successful_pairs], dtype=np.float64)

    # Paired differences and ratios
    time_differences = scalar_times - bit_times
    speedups = scalar_times / bit_times
    flip_ratios = scalar_flips / np.maximum(bit_flips, 1.0)

    # Geometric mean speedup
    geometric_speedup = float(np.exp(np.mean(np.log(speedups))))

    # Cohen's dz
    if n_pairs > 1:
        diff_std = np.std(time_differences, ddof=1)
        cohens_d = np.mean(time_differences) / diff_std if diff_std > 0 else float("inf")
    else:
        cohens_d = float("nan")

    # Wilcoxon signed-rank test
    try:
        wilcoxon_result = stats.wilcoxon(
            scalar_times, bit_times,
            alternative="two-sided", zero_method="wilcox"
        )
        wilcoxon_statistic = float(wilcoxon_result.statistic)
        wilcoxon_pvalue = float(wilcoxon_result.pvalue)
    except Exception:
        wilcoxon_statistic = float("nan")
        wilcoxon_pvalue = float("nan")

    # Bootstrap CIs
    mean_difference_ci = bootstrap_mean_difference(scalar_times, bit_times, n_bootstrap)
    median_difference_ci = bootstrap_median_difference(scalar_times, bit_times, n_bootstrap)

    # Success rates
    success_rates = {
        "scalar": 100.0 * scalar_success_count / total,
        "bit": 100.0 * bit_success_count / total,
        "paired": 100.0 * n_pairs / total,
        "scalar_count": scalar_success_count,
        "bit_count": bit_success_count,
        "paired_count": n_pairs,
        "total": total
    }

    # Descriptive statistics
    scalar_stats = {
        "mean_time": float(np.mean(scalar_times)),
        "median_time": float(np.median(scalar_times)),
        "std_time": float(np.std(scalar_times)),
        "p90_time": float(np.percentile(scalar_times, 90)),
        "p99_time": float(np.percentile(scalar_times, 99)),
        "min_time": float(np.min(scalar_times)),
        "max_time": float(np.max(scalar_times)),
        "mean_flips": float(np.mean(scalar_flips)),
        "median_flips": float(np.median(scalar_flips)),
        "mean_attempts": float(np.mean(scalar_attempts)),
        "median_attempts": float(np.median(scalar_attempts)),
        "throughput": float(n_pairs / np.sum(scalar_times))
    }

    bit_stats = {
        "mean_time": float(np.mean(bit_times)),
        "median_time": float(np.median(bit_times)),
        "std_time": float(np.std(bit_times)),
        "p90_time": float(np.percentile(bit_times, 90)),
        "p99_time": float(np.percentile(bit_times, 99)),
        "min_time": float(np.min(bit_times)),
        "max_time": float(np.max(bit_times)),
        "mean_flips": float(np.mean(bit_flips)),
        "median_flips": float(np.median(bit_flips)),
        "mean_attempts": float(np.mean(bit_attempts)),
        "median_attempts": float(np.median(bit_attempts)),
        "throughput": float(n_pairs / np.sum(bit_times))
    }

    paired_stats = {
        "n_pairs": n_pairs,
        "mean_speedup": float(np.mean(speedups)),
        "median_speedup": float(np.median(speedups)),
        "geometric_mean_speedup": geometric_speedup,
        "std_speedup": float(np.std(speedups)),
        "p90_speedup": float(np.percentile(speedups, 90)),
        "p99_speedup": float(np.percentile(speedups, 99)),
        "mean_time_difference": float(np.mean(time_differences)),
        "median_time_difference": float(np.median(time_differences)),
        "std_time_difference": float(np.std(time_differences, ddof=1)),
        "mean_flip_ratio": float(np.mean(flip_ratios)),
        "median_flip_ratio": float(np.median(flip_ratios)),
        "mean_flip_difference": float(np.mean(scalar_flips - bit_flips)),
        "median_flip_difference": float(np.median(scalar_flips - bit_flips)),
        "cohens_dz": float(cohens_d),
        "wilcoxon_statistic": wilcoxon_statistic,
        "wilcoxon_pvalue": wilcoxon_pvalue,
        "bootstrap_mean_difference_95": mean_difference_ci,
        "bootstrap_median_difference_95": median_difference_ci
    }

    # Heavy-tail ratios
    heavy_tail = {
        "scalar": float(scalar_stats["p99_time"] / scalar_stats["median_time"]),
        "bit": float(bit_stats["p99_time"] / bit_stats["median_time"])
    }

    return {
        "success_rates": success_rates,
        "scalar": scalar_stats,
        "bit": bit_stats,
        "paired": paired_stats,
        "heavy_tail": heavy_tail
    }


# =====================================================================
# PLOTS
# =====================================================================

def save_figure(fig, output_dir, name):
    os.makedirs(output_dir, exist_ok=True)

    pdf_path = os.path.join(output_dir, name + ".pdf")
    png_path = os.path.join(output_dir, name + ".png")

    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"  Saved {pdf_path}")
    print(f"  Saved {png_path}")


def generate_ablation_plots(stats_result, paired_results, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)

    successful = [
        r for r in paired_results
        if r["scalar_success"] and r["bit_success"]
    ]

    if not successful:
        print("No successful paired results for plots.")
        return

    scalar_times = np.asarray([r["scalar_time"] for r in successful])
    bit_times = np.asarray([r["bit_time"] for r in successful])
    speedups = scalar_times / bit_times

    # 1. ECDF
    fig, ax = plt.subplots(figsize=(8, 5))

    scalar_sorted = np.sort(scalar_times)
    bit_sorted = np.sort(bit_times)

    scalar_y = np.arange(1, len(scalar_sorted) + 1) / len(scalar_sorted)
    bit_y = np.arange(1, len(bit_sorted) + 1) / len(bit_sorted)

    ax.step(scalar_sorted, scalar_y, where="post", label="Scalar", linewidth=2)
    ax.step(bit_sorted, bit_y, where="post", label="Bit-Parallel", linewidth=2)

    ax.set_xscale("log")
    ax.set_xlabel("Solve Time (s)")
    ax.set_ylabel("Cumulative Probability")
    ax.set_title("Runtime Distribution (ECDF)")
    ax.grid(True, alpha=0.3)
    ax.legend()

    save_figure(fig, output_dir, "ecdf_runtimes")

    # 2. Paired scatter
    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(scalar_times, bit_times, alpha=0.45, s=20)

    positive_times = np.concatenate([scalar_times, bit_times])
    lower = max(np.min(positive_times) * 0.8, 1e-9)
    upper = np.max(positive_times) * 1.2

    ax.plot([lower, upper], [lower, upper], linestyle="--", linewidth=1, label="y = x")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)

    ax.set_xlabel("Scalar Time (s)")
    ax.set_ylabel("Bit-Parallel Time (s)")
    ax.set_title("Paired Runtime Comparison")
    ax.grid(True, alpha=0.3)
    ax.legend()

    faster = np.sum(bit_times < scalar_times)
    slower_or_equal = len(scalar_times) - faster

    ax.text(
        0.05, 0.95,
        f"Bit faster: {faster}/{len(scalar_times)} ({100*faster/len(scalar_times):.1f}%)\n"
        f"Not faster: {slower_or_equal}/{len(scalar_times)} ({100*slower_or_equal/len(scalar_times):.1f}%)",
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

    save_figure(fig, output_dir, "paired_runtime_scatter")

    # 3. Speedup distribution (sorted)
    fig, ax = plt.subplots(figsize=(8, 5))

    sorted_speedups = np.sort(speedups)

    ax.plot(np.arange(1, len(sorted_speedups) + 1), sorted_speedups, linewidth=1.5)
    ax.axhline(1.0, linestyle="--", linewidth=1, label="1× baseline")

    ax.set_yscale("log")
    ax.set_xlabel("Paired Instance Rank")
    ax.set_ylabel("Speedup (×)")
    ax.set_title("Per-Instance Speedup Distribution")
    ax.grid(True, alpha=0.3)
    ax.legend()

    save_figure(fig, output_dir, "speedup_distribution")

    # 4. Mean/median runtime
    fig, ax = plt.subplots(figsize=(7, 5))

    means = [stats_result["scalar"]["mean_time"], stats_result["bit"]["mean_time"]]
    medians = [stats_result["scalar"]["median_time"], stats_result["bit"]["median_time"]]

    x = np.arange(2)
    width = 0.35

    ax.bar(x - width/2, means, width, label="Mean")
    ax.bar(x + width/2, medians, width, label="Median")

    ax.set_xticks(x)
    ax.set_xticklabels(["Scalar", "Bit-Parallel"])
    ax.set_yscale("log")
    ax.set_ylabel("Solve Time (s)")
    ax.set_title("Mean and Median Runtime")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()

    save_figure(fig, output_dir, "mean_median_runtime")


# =====================================================================
# PRINT SUMMARY
# =====================================================================

def print_summary(stats_result):
    print("\n" + "=" * 72)
    print("ABLATION EXPERIMENT SUMMARY")
    print("=" * 72)

    if "error" in stats_result:
        print(stats_result["error"])
        return

    sr = stats_result["success_rates"]
    scalar = stats_result["scalar"]
    bit = stats_result["bit"]
    paired = stats_result["paired"]
    heavy = stats_result["heavy_tail"]

    print("\nSUCCESS RATE")
    print("-" * 72)
    print(f"Scalar: {sr['scalar_count']}/{sr['total']} ({sr['scalar']:.2f}%)")
    print(f"Bit:    {sr['bit_count']}/{sr['total']} ({sr['bit']:.2f}%)")
    print(f"Paired successful: {sr['paired_count']}/{sr['total']} ({sr['paired']:.2f}%)")

    print("\nRUNTIME")
    print("-" * 72)
    print(f"Mean:   {scalar['mean_time']:.6f}s → {bit['mean_time']:.6f}s = {paired['mean_speedup']:.2f}×")
    print(f"Median: {scalar['median_time']:.6f}s → {bit['median_time']:.6f}s = {paired['median_speedup']:.2f}×")
    print(f"Geometric mean speedup: {paired['geometric_mean_speedup']:.2f}×")
    print(f"P90:    {scalar['p90_time']:.6f}s → {bit['p90_time']:.6f}s = {paired['p90_speedup']:.2f}×")
    print(f"P99:    {scalar['p99_time']:.6f}s → {bit['p99_time']:.6f}s = {paired['p99_speedup']:.2f}×")

    print("\nSEARCH EFFORT")
    print("-" * 72)
    print(f"Mean total flips:")
    print(f"  Scalar: {scalar['mean_flips']:.2f}")
    print(f"  Bit:    {bit['mean_flips']:.2f}")
    print(f"  Ratio:  {paired['mean_flip_ratio']:.2f}×")
    print(f"\nMean attempts:")
    print(f"  Scalar: {scalar['mean_attempts']:.2f}")
    print(f"  Bit:    {bit['mean_attempts']:.2f}")

    print("\nSTATISTICAL ANALYSIS")
    print("-" * 72)
    print(f"Paired Cohen's dz: {paired['cohens_dz']:.4f}")
    print(f"Wilcoxon statistic: {paired['wilcoxon_statistic']:.4f}")
    print(f"Wilcoxon p-value: {paired['wilcoxon_pvalue']:.6e}")

    ci = paired["bootstrap_mean_difference_95"]
    print(f"Bootstrap 95% CI (mean scalar-bit difference): [{ci['lower']:.6f}, {ci['upper']:.6f}] s")

    ci_med = paired["bootstrap_median_difference_95"]
    print(f"Bootstrap 95% CI (median scalar-bit difference): [{ci_med['lower']:.6f}, {ci_med['upper']:.6f}] s")

    print("\nHEAVY TAIL")
    print("-" * 72)
    print(f"Scalar P99 / median: {heavy['scalar']:.2f}")
    print(f"Bit P99 / median:    {heavy['bit']:.2f}")

    print("\n" + "=" * 72)


# =====================================================================
# JSON SERIALIZATION
# =====================================================================

def json_safe(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None
        if np.isinf(obj):
            return str(obj)
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    return obj


# =====================================================================
# MAIN
# =====================================================================

def main():
    print("=" * 72)
    print("CORRECTED CONTROLLED ABLATION")
    print("Hardware-Aligned Bit-Parallel Conflict Evaluation")
    print("=" * 72)

    print(f"\nDataset: {FOLDER_PATH}")
    print(f"Maximum files: {MAX_FILES}")
    print(f"Maximum steps/restart: {MAX_STEPS}")
    print(f"Maximum restarts: {MAX_ATTEMPTS}")
    print(f"Base seed: {BASE_SEED}")

    # Memory analysis
    memory_table = compute_memory_table()

    # Locate first instance
    files = sorted(f for f in os.listdir(FOLDER_PATH) if f.lower().endswith(".cnf"))
    if not files:
        raise FileNotFoundError("No CNF files found.")

    first_instance = SATInstance(os.path.join(FOLDER_PATH, files[0]))

    # Warm-up
    print("\nCompiling and validating kernels...")
    warmup_instance(first_instance)

    popcnt_info = verify_popcnt_assembly()
    print(f"\nPOPCNT assembly check:")
    print(f"  Available: {popcnt_info['assembly_available']}")
    print(f"  POPCNT detected: {popcnt_info['popcnt_detected']}")
    print(f"  {popcnt_info['message']}")

    # Mathematical equivalence test
    selected = first_instance.initialize_state(BASE_SEED)
    bits = first_instance.selected_to_bits(selected)

    energy_ok = validate_state_energy(
        selected, first_instance.flat_lits, first_instance.masks, bits,
        first_instance.num_clauses, first_instance.word_count
    )

    candidate_ok = validate_candidate_evaluations(
        selected, first_instance.flat_lits, first_instance.masks, bits,
        first_instance.num_clauses, first_instance.word_count
    )

    print("\nMathematical equivalence test:")
    print(f"  Total energy: {'PASS' if energy_ok else 'FAIL'}")
    print(f"  Candidate evaluations: {'PASS' if candidate_ok else 'FAIL'}")

    if not energy_ok or not candidate_ok:
        raise RuntimeError("Scalar and bit mathematical objectives are not equivalent.")

    # Microbenchmark
    microbenchmark_result = None
    if RUN_MICROBENCHMARK:
        microbenchmark_result = run_microbenchmark(first_instance, MICROBENCH_EVALS)

    # Full paired ablation
    paired_results, trajectory_checks, popcnt_info = run_ablation_experiment(
        FOLDER_PATH,
        max_files=MAX_FILES,
        max_steps=MAX_STEPS,
        max_attempts=MAX_ATTEMPTS,
        base_seed=BASE_SEED,
        trajectory_test_files=TRAJECTORY_TEST_FILES
    )

    # Statistics
    stats_result = compute_ablation_stats(paired_results, N_BOOTSTRAP)

    # Save JSON
    output = {
        "experiment": {
            "name": "Controlled scalar vs bit-parallel conflict evaluation",
            "canonical_state": "selected[clause] = literal_index",
            "scalar_complexity": "O(k)",
            "bit_complexity": "O(ceil(3k/64))",
            "max_steps": MAX_STEPS,
            "max_attempts": MAX_ATTEMPTS,
            "base_seed": BASE_SEED,
            "trajectory_test_files": TRAJECTORY_TEST_FILES
        },
        "popcnt_verification": popcnt_info,
        "microbenchmark": microbenchmark_result,
        "memory_scaling": memory_table,
        "trajectory_checks": trajectory_checks,
        "statistics": stats_result,
        "paired_results": paired_results
    }

    with open(RESULT_JSON, "w", encoding="utf-8") as f:
        json.dump(json_safe(output), f, indent=2)

    print(f"\nResults written to: {RESULT_JSON}")

    # Plots
    generate_ablation_plots(stats_result, paired_results, OUTPUT_DIR)

    # Final summary
    print_summary(stats_result)

    # Trajectory report
    print("\nTRAJECTORY VERIFICATION")
    print("-" * 72)

    if trajectory_checks:
        passed = sum(1 for x in trajectory_checks if x["identical"])
        print(f"Identical trajectories: {passed}/{len(trajectory_checks)}")

        for check in trajectory_checks:
            print(f"  {check['instance']}: {'PASS' if check['identical'] else 'FAIL'}")

    print("\n" + "=" * 72)
    print("EXPERIMENT COMPLETE")
    print("=" * 72)

    print(
        "\nThe resulting ablation isolates the conflict-evaluation "
        "implementation while keeping the mathematical search state "
        "and search decisions controlled."
    )


if __name__ == "__main__":
    main()

CORRECTED CONTROLLED ABLATION
Hardware-Aligned Bit-Parallel Conflict Evaluation

Dataset: C:\Users\PC\Desktop\JSA\uf20-91
Maximum files: 1000
Maximum steps/restart: 5000
Maximum restarts: 2000
Base seed: 42

MEMORY SCALING ANALYSIS (Capacity-Based Regimes)
NOTE: These are capacity-based estimates, not measured cache misses.
------------------------------------------------------------------------
       n        k        m      Memory KB               Nominal regime
------------------------------------------------------------------------
      20       91        5           1.56                  L1 capacity
      20      200       10           3.12                  L1 capacity
      20      400       19           5.94                  L1 capacity
      20     1000       47          14.69                  L1 capacity
      20     2000       94          29.38                  L1 capacity
      20     5000      235          73.44                  L2 capacity
      20    10000      469     

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
MEMORY SCALING AND CACHE ANALYSIS FOR JSA PAPER REVISION

Generates:
1. Memory scaling curve: Mask memory vs clauses (KB)
2. Cache regime analysis: L1/L2/L3/DRAM thresholds
3. Publication-quality plots for the paper
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import os

# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = "jsa_plots"

# Cache sizes for Intel Core i5-3230M (Ivy Bridge)
L1_SIZE_KB = 32      # 32 KB L1 cache per core
L2_SIZE_KB = 256     # 256 KB L2 cache per core
L3_SIZE_KB = 3072    # 3 MB L3 cache shared

# Color palette (consistent with paper)
COLORS = {
    'L1': '#2ECC71',      # Green
    'L2': '#F1C40F',      # Yellow
    'L3': '#E67E22',      # Orange
    'DRAM': '#E74C3C',    # Red
    'scalar': '#4472C4',
    'bit': '#ED7D31',
    'speedup': '#70AD47',
}

# ============================================================
# MEMORY CALCULATION
# ============================================================

def compute_mask_memory(n, k):
    """
    Compute contradiction mask memory footprint.
    
    Formula from the paper:
    M(n,k) = 2n * ceil(3k/64) * 8 bytes
    
    Args:
        n: number of variables
        k: number of clauses
    
    Returns:
        memory_bytes: total memory in bytes
        memory_kb: total memory in KB
        words: number of 64-bit words per mask
    """
    words = (3 * k + 63) // 64  # ceil(3k/64)
    memory_bytes = 2 * n * words * 8
    memory_kb = memory_bytes / 1024.0
    return memory_bytes, memory_kb, words


def get_cache_regime(memory_bytes):
    """
    Determine cache regime based on memory footprint.
    
    Returns:
        regime: 'L1', 'L2', 'L3', or 'DRAM'
    """
    if memory_bytes <= L1_SIZE_KB * 1024:
        return 'L1'
    elif memory_bytes <= L2_SIZE_KB * 1024:
        return 'L2'
    elif memory_bytes <= L3_SIZE_KB * 1024:
        return 'L3'
    else:
        return 'DRAM'


# ============================================================
# DATA GENERATION
# ============================================================

def generate_memory_data():
    """Generate comprehensive memory scaling data."""
    
    # Test configurations
    test_cases = [
        # (n, k, label)
        (20, 91, 'uf20-91 (evaluated)'),
        (20, 200, ''),
        (20, 400, ''),
        (20, 1000, ''),
        (20, 2000, ''),
        (20, 5000, ''),
        (20, 10000, ''),
        (100, 425, ''),
        (100, 850, ''),
        (100, 2125, ''),
        (200, 850, ''),
        (200, 2125, ''),
        (200, 4250, ''),
        (500, 4250, ''),
        (1000, 4250, ''),
    ]
    
    results = []
    for n, k, label in test_cases:
        memory_bytes, memory_kb, words = compute_mask_memory(n, k)
        regime = get_cache_regime(memory_bytes)
        results.append({
            'n': n,
            'k': k,
            'words': words,
            'memory_bytes': memory_bytes,
            'memory_kb': memory_kb,
            'regime': regime,
            'label': label
        })
    
    return results


def generate_scaling_curves():
    """Generate data for scaling curves."""
    
    # Fixed n=20, vary k
    n_fixed = 20
    k_values = list(range(91, 10001, 100))  # 91 to 10000 in steps of 100
    
    data = []
    for k in k_values:
        memory_bytes, memory_kb, words = compute_mask_memory(n_fixed, k)
        regime = get_cache_regime(memory_bytes)
        data.append({
            'k': k,
            'memory_kb': memory_kb,
            'regime': regime,
            'words': words
        })
    
    return data


# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def format_kb(x, pos):
    """Format KB values with appropriate units."""
    if x >= 1024:
        return f'{x/1024:.1f} MB'
    else:
        return f'{x:.0f} KB'


def plot_memory_scaling(memory_data, scaling_data, output_dir=OUTPUT_DIR):
    """
    Generate publication-quality memory scaling plot.
    
    Figure shows:
    1. Memory footprint vs clause count (log-log scale)
    2. Cache regime regions (L1, L2, L3, DRAM)
    3. Markers for evaluated configurations
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # ------------------------------------------------------------
    # 1. Cache regime regions (shaded background)
    # ------------------------------------------------------------
    
    # L1 region (0 to 32 KB)
    ax.axhspan(0, L1_SIZE_KB, alpha=0.15, color=COLORS['L1'], label='L1 (32 KB)')
    
    # L2 region (32 to 256 KB)
    ax.axhspan(L1_SIZE_KB, L2_SIZE_KB, alpha=0.15, color=COLORS['L2'], label='L2 (256 KB)')
    
    # L3 region (256 KB to 3 MB)
    ax.axhspan(L2_SIZE_KB, L3_SIZE_KB, alpha=0.15, color=COLORS['L3'], label='L3 (3 MB)')
    
    # DRAM region (>3 MB)
    ax.axhspan(L3_SIZE_KB, max([d['memory_kb'] for d in scaling_data]) * 1.1, 
               alpha=0.15, color=COLORS['DRAM'], label='DRAM')
    
    # ------------------------------------------------------------
    # 2. Scaling curve (n=20, varying k)
    # ------------------------------------------------------------
    
    k_vals = [d['k'] for d in scaling_data]
    mem_vals = [d['memory_kb'] for d in scaling_data]
    regimes = [d['regime'] for d in scaling_data]
    
    # Color by regime
    regime_colors = [COLORS[r] for r in regimes]
    
    ax.plot(k_vals, mem_vals, 'o-', color='#2C3E50', linewidth=2, 
            markersize=6, label='Mask memory (n=20)')
    
    # Color markers by regime
    for k, mem, regime in zip(k_vals, mem_vals, regimes):
        ax.plot(k, mem, 'o', color=COLORS[regime], markersize=8, 
                markeredgecolor='black', markeredgewidth=0.5)
    
    # ------------------------------------------------------------
    # 3. Mark evaluated configuration (n=20, k=91)
    # ------------------------------------------------------------
    
    eval_k = 91
    eval_mem = compute_mask_memory(20, eval_k)[1]
    ax.plot(eval_k, eval_mem, 's', color='red', markersize=12, 
            markeredgecolor='black', markeredgewidth=1.5,
            label=f'Evaluated: n=20, k={eval_k} ({eval_mem:.1f} KB)')
    
    # ------------------------------------------------------------
    # 4. Annotations for key thresholds
    # ------------------------------------------------------------
    
    # L1 threshold
    ax.axhline(y=L1_SIZE_KB, color=COLORS['L1'], linestyle='--', alpha=0.5, linewidth=1)
    ax.text(5500, L1_SIZE_KB + 8, f'L1: {L1_SIZE_KB} KB', 
            fontsize=10, color=COLORS['L1'], fontweight='bold')
    
    # L2 threshold
    ax.axhline(y=L2_SIZE_KB, color=COLORS['L2'], linestyle='--', alpha=0.5, linewidth=1)
    ax.text(5500, L2_SIZE_KB + 20, f'L2: {L2_SIZE_KB} KB', 
            fontsize=10, color=COLORS['L2'], fontweight='bold')
    
    # L3 threshold
    ax.axhline(y=L3_SIZE_KB, color=COLORS['L3'], linestyle='--', alpha=0.5, linewidth=1)
    ax.text(5500, L3_SIZE_KB + 30, f'L3: {L3_SIZE_KB} KB (3 MB)', 
            fontsize=10, color=COLORS['L3'], fontweight='bold')
    
    # ------------------------------------------------------------
    # 5. Formatting
    # ------------------------------------------------------------
    
    ax.set_xlabel('Number of Clauses (K)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mask Memory (KB)', fontsize=12, fontweight='bold')
    ax.set_title('Contradiction Mask Memory Scaling\n' + 
                 r'$M(n,K) = 2n \times \lceil 3K/64 \rceil \times 8$ bytes',
                 fontsize=14, fontweight='bold')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    ax.grid(True, alpha=0.3, linestyle='--')
    
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    
    # Format y-axis ticks
    ax.yaxis.set_major_formatter(FuncFormatter(format_kb))
    
    # Set axis limits
    ax.set_xlim(80, 12000)
    ax.set_ylim(0.8, max(mem_vals) * 1.2)
    
    plt.tight_layout()
    
    # Save
    plt.savefig(f'{output_dir}/memory_scaling_curve.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(f'{output_dir}/memory_scaling_curve.png', bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"  ✓ memory_scaling_curve.pdf/png")
    
    return fig


def plot_cache_regime_pie(memory_data, output_dir=OUTPUT_DIR):
    """Generate pie chart showing cache regime distribution."""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Count regimes
    regime_counts = {'L1': 0, 'L2': 0, 'L3': 0, 'DRAM': 0}
    for d in memory_data:
        regime_counts[d['regime']] += 1
    
    labels = []
    sizes = []
    colors = []
    for regime in ['L1', 'L2', 'L3', 'DRAM']:
        if regime_counts[regime] > 0:
            labels.append(f'{regime} ({regime_counts[regime]})')
            sizes.append(regime_counts[regime])
            colors.append(COLORS[regime])
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    wedges, texts, autotexts = ax.pie(
        sizes, 
        labels=labels, 
        colors=colors,
        autopct='%1.1f%%',
        startangle=90,
        explode=[0.02] * len(sizes),
        shadow=True,
        textprops={'fontsize': 12}
    )
    
    ax.set_title('Cache Regime Distribution\n(All Test Configurations)', 
                 fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    plt.savefig(f'{output_dir}/cache_regime_pie.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(f'{output_dir}/cache_regime_pie.png', bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"  ✓ cache_regime_pie.pdf/png")
    
    return fig


def plot_memory_table(memory_data, output_dir=OUTPUT_DIR):
    """Generate a formatted table image for the paper."""
    
    os.makedirs(output_dir, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('tight')
    ax.axis('off')
    
    # Prepare table data
    table_data = [
        ['Variables (n)', 'Clauses (K)', 'Words (m)', 'Memory (KB)', 'Cache Regime']
    ]
    
    for d in memory_data:
        regime_emoji = {'L1': '🟢', 'L2': '🟡', 'L3': '🟠', 'DRAM': '🔴'}.get(d['regime'], '')
        table_data.append([
            str(d['n']),
            str(d['k']),
            str(d['words']),
            f"{d['memory_kb']:.2f}",
            f"{regime_emoji} {d['regime']}"
        ])
    
    # Create table
    table = ax.table(
        cellText=table_data,
        loc='center',
        cellLoc='center',
        colWidths=[0.12, 0.12, 0.10, 0.15, 0.15]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.5)
    
    # Style header
    for i in range(len(table_data[0])):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(color='white', weight='bold')
    
    # Style rows by regime
    regime_colors = {'L1': '#E8F5E9', 'L2': '#FFF3E0', 'L3': '#FFE0B2', 'DRAM': '#FFCDD2'}
    for row in range(1, len(table_data)):
        regime = memory_data[row-1]['regime']
        for col in range(len(table_data[0])):
            table[(row, col)].set_facecolor(regime_colors.get(regime, 'white'))
    
    plt.title('Contradiction Mask Memory Scaling Table', fontsize=14, pad=20, fontweight='bold')
    plt.tight_layout()
    
    plt.savefig(f'{output_dir}/memory_scaling_table.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(f'{output_dir}/memory_scaling_table.png', bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"  ✓ memory_scaling_table.pdf/png")
    
    return fig


def plot_cache_failure_risk(output_dir=OUTPUT_DIR):
    """Generate cache failure risk visualization."""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Data for cache failure analysis
    scenarios = [
        {'n': 20, 'k': 91, 'label': 'Evaluated (n=20, k=91)', 'memory_kb': 1.56},
        {'n': 20, 'k': 1000, 'label': 'n=20, k=1000', 'memory_kb': 14.69},
        {'n': 20, 'k': 5000, 'label': 'n=20, k=5000', 'memory_kb': 73.44},
        {'n': 100, 'k': 850, 'label': 'n=100, k=850', 'memory_kb': 62.50},
        {'n': 200, 'k': 2125, 'label': 'n=200, k=2125', 'memory_kb': 312.50},
        {'n': 500, 'k': 4250, 'label': 'n=500, k=4250', 'memory_kb': 1562.50},
        {'n': 1000, 'k': 4250, 'label': 'n=1000, k=4250', 'memory_kb': 3125.00},
    ]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create horizontal bar chart
    y_pos = np.arange(len(scenarios))
    memory_kb = [s['memory_kb'] for s in scenarios]
    labels = [s['label'] for s in scenarios]
    
    # Determine colors based on regime
    colors = []
    for s in scenarios:
        regime = get_cache_regime(s['memory_kb'] * 1024)
        colors.append(COLORS[regime])
    
    bars = ax.barh(y_pos, memory_kb, color=colors, edgecolor='black', linewidth=0.8)
    
    # Add cache regime annotations
    ax.axvline(x=L1_SIZE_KB, color=COLORS['L1'], linestyle='--', linewidth=1.5, label='L1: 32 KB')
    ax.axvline(x=L2_SIZE_KB, color=COLORS['L2'], linestyle='--', linewidth=1.5, label='L2: 256 KB')
    ax.axvline(x=L3_SIZE_KB, color=COLORS['L3'], linestyle='--', linewidth=1.5, label='L3: 3 MB')
    
    ax.set_xlabel('Mask Memory (KB)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Configuration', fontsize=12, fontweight='bold')
    ax.set_title('Cache Failure Risk Analysis', fontsize=14, fontweight='bold')
    ax.set_xscale('log')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, memory_kb):
        if val >= 1024:
            label = f'{val/1024:.2f} MB'
        else:
            label = f'{val:.2f} KB'
        ax.text(bar.get_width() * 1.05, bar.get_y() + bar.get_height()/2, 
                label, va='center', fontsize=9)
    
    plt.tight_layout()
    
    plt.savefig(f'{output_dir}/cache_failure_risk.pdf', bbox_inches='tight', dpi=300)
    plt.savefig(f'{output_dir}/cache_failure_risk.png', bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"  ✓ cache_failure_risk.pdf/png")
    
    return fig


# ============================================================
# PRINT TABLE FOR PAPER
# ============================================================

def print_memory_table_latex(memory_data):
    """Generate LaTeX table for the paper."""
    
    print("\n" + "=" * 70)
    print("MEMORY SCALING TABLE (LaTeX Format)")
    print("=" * 70)
    
    print(r"""
\begin{table}[H]
\centering
\small
\caption{Contradiction Mask Memory Scaling (Capacity-Based Regimes)}
\begin{tabular}{lcccc}
\toprule
\textbf{Variables ($n$)} & \textbf{Clauses ($K$)} & \textbf{Words ($m$)} & \textbf{Memory (KB)} & \textbf{Cache Regime} \\
\midrule""")
    
    for d in memory_data:
        # Skip entries with empty labels
        if d['label']:
            print(f"{d['n']:>8} & {d['k']:>8} & {d['words']:>8} & {d['memory_kb']:>10.2f} & {d['regime']} \\\\")
    
    print(r"""
\bottomrule
\end{tabular}
\label{tab:memory}
\end{table}
""")
    
    print("\n" + "=" * 70)


def print_cache_analysis_text():
    """Generate cache analysis text for the paper."""
    
    print("\n" + "=" * 70)
    print("CACHE ANALYSIS TEXT FOR PAPER")
    print("=" * 70)
    
    print(r"""
\subsection{Memory Scaling and Cache Behavior}

For a SAT instance with $n$ variables and $K$ clauses, the contradiction mask memory footprint is:

\begin{equation}
M(n,K) = 2n \times \left\lceil \frac{3K}{64} \right\rceil \times 8 \text{ bytes}
\label{eq:memory}
\end{equation}

\noindent where $\lceil 3K/64 \rceil$ is the number of 64-bit words per mask.

\textbf{Cache Behavior Analysis:}

\begin{itemize}
\item \textbf{L1 (32 KB):} For $n=20$, configurations up to $K \approx 2{,}000$ remain L1-resident (29.38 KB), enabling 1-3 cycle access latency. This includes the evaluated uf20-91 configuration ($K=91$, 1.56 KB).

\item \textbf{L2 (256 KB):} For $n=20$, the mask table fits in L2 up to $K \approx 10{,}000$ (146.56 KB). For $n=100$, L2 capacity handles up to $K \approx 2{,}125$ (156.25 KB). L2 access latency is approximately 10-20 cycles.

\item \textbf{L3 (3 MB):} For $n=200$, up to $K \approx 4{,}250$ (625 KB) fits in L3. For $n=500$, up to $K \approx 4{,}250$ (1,562.50 KB) fits in L3. L3 access latency is approximately 40-60 cycles.

\item \textbf{DRAM regime:} When the mask table exceeds L3 cache (e.g., $n=1{,}000$, $K=4{,}250$ at 3,125 KB), DRAM accesses are required at 80-100 ns ($\sim$200 cycles). This reduces the theoretical speedup from $26\times$ to approximately $8-10\times$, still providing significant benefit.
\end{itemize}

\textbf{Note:} These are capacity-based estimates, not measured cache-miss data. Actual cache behavior depends on access patterns, prefetching, and contention. Hardware performance counters are required for precise cache-miss measurement.
""")
    
    print("\n" + "=" * 70)


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 70)
    print("MEMORY SCALING AND CACHE ANALYSIS")
    print("Hardware-Aligned Bit-Parallel Conflict Evaluation")
    print("=" * 70)
    
    # Generate data
    memory_data = generate_memory_data()
    scaling_data = generate_scaling_curves()
    
    # Print data table
    print("\nMemory Scaling Data:")
    print("-" * 70)
    print(f"{'n':>8} {'k':>8} {'words':>8} {'Memory (KB)':>14} {'Regime':>10}")
    print("-" * 70)
    for d in memory_data:
        print(f"{d['n']:>8} {d['k']:>8} {d['words']:>8} {d['memory_kb']:>14.2f} {d['regime']:>10}")
    
    # Generate plots
    print("\n" + "-" * 70)
    print("Generating Publication-Quality Plots...")
    print("-" * 70)
    
    plot_memory_scaling(memory_data, scaling_data)
    plot_cache_regime_pie(memory_data)
    plot_memory_table(memory_data)
    plot_cache_failure_risk()
    
    # Print LaTeX table
    print_memory_table_latex(memory_data)
    
    # Print cache analysis text
    print_cache_analysis_text()
    
    print("\n" + "=" * 70)
    print("✅ COMPLETE! Plots saved to jsa_plots/")
    print("=" * 70)
    print("\nGenerated files:")
    print("  ├── memory_scaling_curve.pdf/png")
    print("  ├── cache_regime_pie.pdf/png")
    print("  ├── memory_scaling_table.pdf/png")
    print("  └── cache_failure_risk.pdf/png")


if __name__ == "__main__":
    main()

MEMORY SCALING AND CACHE ANALYSIS
Hardware-Aligned Bit-Parallel Conflict Evaluation

Memory Scaling Data:
----------------------------------------------------------------------
       n        k    words    Memory (KB)     Regime
----------------------------------------------------------------------
      20       91        5           1.56         L1
      20      200       10           3.12         L1
      20      400       19           5.94         L1
      20     1000       47          14.69         L1
      20     2000       94          29.38         L1
      20     5000      235          73.44         L2
      20    10000      469         146.56         L2
     100      425       20          31.25         L1
     100      850       40          62.50         L2
     100     2125      100         156.25         L2
     200      850       40         125.00         L2
     200     2125      100         312.50         L3
     200     4250      200         625.00         L3
     500  

C:\Users\PC\AppData\Local\Temp\ipykernel_5824\1496123048.py:372: UserWarning: Glyph 128994 (\N{LARGE GREEN CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(f'{output_dir}/memory_scaling_table.pdf', bbox_inches='tight', dpi=300)
C:\Users\PC\AppData\Local\Temp\ipykernel_5824\1496123048.py:372: UserWarning: Glyph 128993 (\N{LARGE YELLOW CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(f'{output_dir}/memory_scaling_table.pdf', bbox_inches='tight', dpi=300)
C:\Users\PC\AppData\Local\Temp\ipykernel_5824\1496123048.py:372: UserWarning: Glyph 128992 (\N{LARGE ORANGE CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(f'{output_dir}/memory_scaling_table.pdf', bbox_inches='tight', dpi=300)
C:\Users\PC\AppData\Local\Temp\ipykernel_5824\1496123048.py:372: UserWarning: Glyph 128308 (\N{LARGE RED CIRCLE}) missing from font(s) DejaVu Sans.
  plt.savefig(f'{output_dir}/memory_scaling_table.pdf', bbox_inches='tight', dpi=300)
C:\Users\PC\AppData\Local\Temp\ipykernel_5824\1496123048

  ✓ memory_scaling_table.pdf/png
  ✓ cache_failure_risk.pdf/png

MEMORY SCALING TABLE (LaTeX Format)

\begin{table}[H]
\centering
\small
\caption{Contradiction Mask Memory Scaling (Capacity-Based Regimes)}
\begin{tabular}{lcccc}
\toprule
\textbf{Variables ($n$)} & \textbf{Clauses ($K$)} & \textbf{Words ($m$)} & \textbf{Memory (KB)} & \textbf{Cache Regime} \\
\midrule
      20 &       91 &        5 &       1.56 & L1 \\

\bottomrule
\end{tabular}
\label{tab:memory}
\end{table}



CACHE ANALYSIS TEXT FOR PAPER

\subsection{Memory Scaling and Cache Behavior}

For a SAT instance with $n$ variables and $K$ clauses, the contradiction mask memory footprint is:

\begin{equation}
M(n,K) = 2n \times \left\lceil \frac{3K}{64} \right\rceil \times 8 \text{ bytes}
\label{eq:memory}
\end{equation}

\noindent where $\lceil 3K/64 \rceil$ is the number of 64-bit words per mask.

\textbf{Cache Behavior Analysis:}

\begin{itemize}
\item \textbf{L1 (32 KB):} For $n=20$, configurations up to $K \approx 2{,}0